# Random Agent Baseline
Runs the random policy over N episodes and measures reward.

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
from src.env.warehouse_env import WarehouseEnv
from src.analytics import RewardTracker

with open('../configs/env_config.yaml') as f:
    config = yaml.safe_load(f)

env = WarehouseEnv(config)
print(f"Env: {config['env']['name']}  |  Agents: {env.n_agents}  |  Action dim: {env.action_dim}")

In [ ]:
def random_policy(obs, action_dim, env):
    ACTION_INTERACT = 4
    actions = [np.random.randint(action_dim) for _ in obs]
    u = env._env.unwrapped
    goal_set = {(gc, gr) for gc, gr in u.goals}
    shelf_set = {(s.x, s.y) for s in u.shelfs}
    for i, agent in enumerate(u.agents):
        pos = (agent.x, agent.y)
        if agent.carrying_shelf and pos in goal_set:
            actions[i] = ACTION_INTERACT
        elif agent.carrying_shelf:
            actions[i] = np.random.randint(4)
        elif not agent.carrying_shelf and pos in goal_set:
            actions[i] = np.random.randint(4)
        elif not agent.carrying_shelf and pos in shelf_set:
            actions[i] = ACTION_INTERACT
    return actions

In [ ]:
N_EPISODES = 10000
max_steps = config['env'].get('max_steps', 500)
tracker = RewardTracker(n_agents=env.n_agents)

for ep in range(N_EPISODES):
    obs = env.reset()
    tracker.start_episode()
    for _ in range(max_steps):
        actions = random_policy(obs, env.action_dim, env)
        obs, rews, dones, _ = env.step(actions)
        tracker.record_step(rews)
        if all(dones):
            break
    tracker.end_episode()
    ep_data = tracker.episodes[-1]
    print(f"  ep {ep+1:3d}/{N_EPISODES}  steps={ep_data['n_steps']:4d}  team_reward={ep_data['team_total_reward']:.3f}")

env.close()

In [ ]:
summary = tracker.summary()
print('--- Random Baseline Summary ---')
print(f"  Episodes : {summary['n_episodes']}")
print(f"  Team total reward  — mean: {summary['team_total_reward']['mean']:.4f}  "
      f"std: {summary['team_total_reward']['std']:.4f}  "
      f"[{summary['team_total_reward']['min']:.4f}, {summary['team_total_reward']['max']:.4f}]")
print(f"  Episode length     — mean: {summary['episode_length']['mean']:.1f}")
for i, v in enumerate(summary['per_agent_mean_total']):
    print(f"  Agent {i} mean total reward: {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt

rewards = [ep['team_total_reward'] for ep in tracker.episodes]
plt.figure(figsize=(10, 4))
plt.bar(range(len(rewards)), rewards, color='steelblue', alpha=0.7)
plt.axhline(summary['team_total_reward']['mean'], color='red', linestyle='--', label=f"mean={summary['team_total_reward']['mean']:.3f}")
plt.xlabel('Episode')
plt.ylabel('Team Total Reward')
plt.title('Random Agent Baseline — Reward per Episode')
plt.legend()
plt.tight_layout()
plt.savefig('../results/plots/random_baseline_reward.png', dpi=150)
plt.show()

In [ ]:
tracker.save('../results/logs/random_baseline_rewards.json')
tracker.save_csv('../results/logs/random_baseline_rewards.csv')